# SQL Agent with Walmart Sales
**Goal:** Create a basic SQL agent to interact with the Walmart Sales database

## Libraries

In [1]:
# Most Important: AI
from langchain_openai import ChatOpenAI
from langchain_classic.chains import create_sql_query_chain
from langchain_community.utilities import SQLDatabase

# Data Science
import pandas as pd
import sqlalchemy as sql

# Utilities
import os
import yaml
import re
from pprint import pprint
from IPython.display import Markdown

C:\Users\User\AppData\Local\Temp\ipykernel_23304\684731721.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


## AI Setup

In [2]:
os.environ["OPENAI_API_KEY"] = yaml.safe_load(open('../credentials.yml'))['openai']

OPENAI_LLM = "gpt-4o-mini"

## 1.0 Database Setup — Walmart Sales

In [3]:
PATH_DB = "sqlite:///../data/walmart_sales.db"

sql_engine = sql.create_engine(PATH_DB)
conn = sql_engine.connect()

# Show all tables
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)

,name
0,daily_demand
1,sqlite_stat1
2,sqlite_stat4


## 2.0 Connect LangChain to the Database

In [4]:
db = SQLDatabase.from_uri(PATH_DB)

print("Dialect:", db.dialect)
print("Tables:", db.get_usable_table_names())
print("\nSample data:")
print(db.run("SELECT * FROM daily_demand LIMIT 5;"))

Dialect: sqlite
Tables: ['daily_demand']

Sample data:
[('FOODS_3_090', 1046, '2011-01-29'), ('FOODS_3_090', 1036, '2011-01-30'), ('FOODS_3_090', 673, '2011-01-31'), ('FOODS_3_090', 642, '2011-02-01'), ('FOODS_3_090', 531, '2011-02-02')]


## 3.0 Create the SQL Query Chain (Agent)

In [5]:
model = ChatOpenAI(
    model=OPENAI_LLM,
    temperature=0.7,
)

# Create the SQL query chain
chain = create_sql_query_chain(model, db)
chain

RunnableAssign(mapper={
  input: RunnableLambda(...),
  table_info: RunnableLambda(...)
})
| RunnableLambda(lambda x: {k: v for k, v in x.items() if k not in ('question', 'table_names_to_use')})
| PromptTemplate(input_variables=['input', 'table_info'], input_types={}, partial_variables={'top_k': '5'}, template='You are a SQLite expert. Given an input question, first create a syntactically correct SQLite query to run, then look at the results of the query and return the answer to the input question.\nUnless the user specifies in the question a specific number of examples to obtain, query for at most {top_k} results using the LIMIT clause as per SQLite. You can order the results to return the most informative data in the database.\nNever query for all columns from a table. You must query only the columns that are needed to answer the question. Wrap each column name in double quotes (") to denote them as delimited identifiers.\nPay attention to use only the column names you can see in the

In [6]:
response = chain.invoke({'question': "What are the top 10 items by total cumulative demand value?"})
pprint(response)
Markdown(response)

('SQLQuery: SELECT "item_id", SUM("value") AS "total_value" FROM daily_demand '
 'GROUP BY "item_id" ORDER BY "total_value" DESC LIMIT 5')


SQLQuery: SELECT "item_id", SUM("value") AS "total_value" FROM daily_demand GROUP BY "item_id" ORDER BY "total_value" DESC LIMIT 5

## 4.0 SQL Parsing Utility

In [7]:
def extract_sql_code(text: str):
    """Extract the SQL query from an LLM response. Returns None if not found."""
    if not text:
        return None
    for pat in [
        r"SQLQuery:\s*```sql\s*([\s\S]+?)```",
        r"```sql\s*([\s\S]+?)```",
        r"```[\w]*\s*(SELECT[\s\S]+?)```",
        r"SQLQuery:\s*(SELECT[\s\S]+?)(?:\n\n|$)",
        r"(SELECT[\s\S]+?)(?:;|\n\n|$)",
    ]:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            return m.group(1).strip().rstrip(";")
    return None

pprint(extract_sql_code(response))

('SELECT "item_id", SUM("value") AS "total_value" FROM daily_demand GROUP BY '
 '"item_id" ORDER BY "total_value" DESC LIMIT 5')


In [8]:
# Run extracted SQL against DB
pprint(db.run(extract_sql_code(response)))

("[('FOODS_3_090', 1002529), ('FOODS_3_586', 920242), ('FOODS_3_252', 565299), "
 "('FOODS_3_555', 491287), ('FOODS_3_714', 396172)]")


## 5.0 Additional Questions

In [9]:
# Total demand value aggregated by year-month
q = chain.invoke({'question': "What is the total demand value by year-month? Order results chronologically."})
sql_q = extract_sql_code(q)
pprint(sql_q)
pd.read_sql(sql_q, conn)

('SELECT strftime(\'%Y-%m\', "date") AS "year_month", SUM("value") AS '
 '"total_demand"\n'
 'FROM daily_demand\n'
 'GROUP BY "year_month"\n'
 'ORDER BY "year_month" ASC\n'
 'LIMIT 5')


,year_month,total_demand
0,2011-01,9758
1,2011-02,69217
2,2011-03,64767
3,2011-04,69084
4,2011-05,69958


In [10]:
# Top items by average daily demand
q2 = chain.invoke({'question': "Which 10 items have the highest average daily demand value?"})
sql_q2 = extract_sql_code(q2)
pprint(sql_q2)
pd.read_sql(sql_q2, conn)

('SELECT "item_id", AVG("value") AS "average_demand"\n'
 'FROM daily_demand\n'
 'GROUP BY "item_id"\n'
 'ORDER BY "average_demand" DESC\n'
 'LIMIT 10')


,item_id,average_demand
0,FOODS_3_090,524.061160
1,FOODS_3_586,481.046524
2,FOODS_3_252,295.503921
3,FOODS_3_555,256.814950
4,FOODS_3_714,207.094616
5,FOODS_3_587,207.066911
6,FOODS_3_694,203.868792
7,FOODS_3_226,189.797177
8,FOODS_3_202,154.568217
9,FOODS_3_723,148.631992


In [11]:
conn.close()